# 02 - Raw BLD versus Fixed and Adaptive CCI

This standalone smile notebook assumes the reviewed semantic policy from graph discovery: mouth plus both lips. It selects its own eligible cohort and compares identical model, prompt, seed, mask, and scheduler settings. `disabled` is raw BLD without CCI updates, `fixed_equal` uses CCI with fixed constraint coefficients, and `feedback` enables adaptive dual weighting. The experiment entry point runs in-process so progress is visible in the notebook output.

In [ ]:
from pathlib import Path
import json, logging, os, runpy, subprocess, sys, time
os.environ['PYTHONUNBUFFERED'] = '1'
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s', force=True)

REPO_URL = 'https://github.com/lokissdo/cci-diff.git'
GIT_REF = 'main'
PROJECT_ROOT = Path('/kaggle/working/cci-diff')
ASSET_ROOT = Path('/kaggle/input/cci-assets')
DATA_ROOT = Path('/kaggle/input/celebamask-hq/CelebAMask-HQ')
MODEL_PATH = 'sd2-community/stable-diffusion-2-1'
CLASSIFIER_PATH = ASSET_ROOT / 'resnet50_multilabel_model.pth'
IDENTITY_MODEL_PATH = ASSET_ROOT / 'facenet_vggface2.ts'
IMAGE_ROOT = DATA_ROOT / 'CelebA-HQ-img'
MASK_ROOT = DATA_ROOT / 'CelebAMask-HQ-mask-anno'
OUTPUT_ROOT = Path('/kaggle/working/cci_fixed_vs_adaptive')

DEVICE = 'cuda'
SAMPLE_COUNT = 300
NUM_INFERENCE_STEPS = 35
SEED = 42
ASSUMED_REGIONS = {'smile': ['mouth', 'upper_lip', 'lower_lip']}

In [ ]:
if not (PROJECT_ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(PROJECT_ROOT)], check=True)
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', GIT_REF], cwd=PROJECT_ROOT, check=True)
subprocess.run(['git', 'checkout', '--force', '--detach', 'FETCH_HEAD'], cwd=PROJECT_ROOT, check=True)
resolved_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True).strip()
runtime_packages = ['diffusers', 'transformers', 'accelerate', 'safetensors', 'open-clip-torch', 'grad-cam']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *runtime_packages], check=True)
if not IMAGE_ROOT.is_dir():
    image_roots = sorted(Path('/kaggle/input').rglob('CelebA-HQ-img'))
    if len(image_roots) != 1:
        raise FileNotFoundError(f'Expected one CelebA-HQ-img directory, found: {image_roots}')
    IMAGE_ROOT = image_roots[0]
if not MASK_ROOT.is_dir():
    mask_roots = sorted(Path('/kaggle/input').rglob('CelebAMask-HQ-mask-anno'))
    if len(mask_roots) != 1:
        raise FileNotFoundError(f'Expected one CelebAMask-HQ-mask-anno directory, found: {mask_roots}')
    MASK_ROOT = mask_roots[0]
required = [CLASSIFIER_PATH, IDENTITY_MODEL_PATH, IMAGE_ROOT, MASK_ROOT]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Update the configuration paths; missing: ' + ', '.join(missing))
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT), '--no-deps'], check=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Assumed regions:', ASSUMED_REGIONS, 'source commit:', resolved_commit)

In [ ]:
# The pilot's smile task binds generation to mouth + both lips.
arguments = [
    '--features', 'smile', '--limit', str(SAMPLE_COUNT),
    '--controller_modes', 'disabled', 'fixed_equal', 'feedback',
    '--model_path', str(MODEL_PATH), '--classifier_path', str(CLASSIFIER_PATH),
    '--allow_model_download',
    '--identity_model_path', str(IDENTITY_MODEL_PATH),
    '--image_root', str(IMAGE_ROOT), '--mask_root', str(MASK_ROOT),
    '--device', DEVICE, '--torch_dtype', 'auto',
    '--python_executable', sys.executable,
    '--seed', str(SEED), '--num_inference_steps', str(NUM_INFERENCE_STEPS),
    '--mask_shapes', '4,4,3', '--continue_on_error',
    '--output_dir', str(OUTPUT_ROOT),
]
script = PROJECT_ROOT / 'scripts/run_clean_cci_pilot.py'
previous_argv = sys.argv[:]
sys.argv = [str(script), *arguments]
print(f'[{time.strftime("%H:%M:%S")}] START smile raw/fixed/adaptive evaluation', flush=True)
print(' '.join(sys.argv), flush=True)
try:
    try:
        runpy.run_path(str(script), run_name='__main__')
    except SystemExit as error:
        if error.code not in (None, 0):
            raise
finally:
    sys.argv = previous_argv
print(f'[{time.strftime("%H:%M:%S")}] DONE smile raw/fixed/adaptive evaluation', flush=True)

In [ ]:
import pandas as pd
results = pd.read_csv(OUTPUT_ROOT / 'pilot_results.csv')
results['controller_mode'] = results['variant'].map({'A0': 'raw_bld', 'A2': 'fixed_equal', 'A3': 'feedback'})
display(results.groupby(['feature', 'controller_mode']).agg(
    count=('target_pass', 'size'),
    generation_classifier_fr=('target_pass', 'mean'),
    desired_probability=('desired_probability', 'mean'),
    identity_cosine=('identity_cosine', 'mean'),
    non_target_drift=('non_target_drift', 'mean'),
    changed_fraction_5=('changed_fraction_5', 'mean'),
    outside_semantic_fraction_5=('outside_semantic_fraction_5', 'mean'),
    residual_tv=('residual_tv', 'mean'),
    runtime_seconds=('runtime_seconds', 'mean'),
).reset_index())
print('Source/output comparisons:', OUTPUT_ROOT / 'comparisons')